# Deploying Gemma 4 E4B Offline for Rural Indian Healthcare
## ASHA Sahayak — आशा सहायक

> **An offline-first multimodal AI triage assistant for India's 1 million+ ASHA community health workers.**

---

### The Problem

India has over **1 million ASHA (Accredited Social Health Activists) workers** — government-trained rural community health workers who are the first point of contact for healthcare in thousands of villages.

They face a brutal set of constraints:

| Challenge | Reality |
|---|---|
| Connectivity | No reliable 4G in many villages |
| Distance to care | Nearest doctor can be 20+ km away |
| Language | Workers speak only Marathi, Hindi, or other regional languages |
| Training | Class 10 education + short health course |
| Privacy | India's DPDP Act prohibits uploading patient health photos to the cloud |

**Cloud AI is not an option.** What these workers need is a model that:
- Runs **fully offline** on a basic Android phone or laptop
- Understands **photographs** of wounds, rashes, burns
- Responds in **Marathi** (or Hindi) — not English
- Costs **nothing to run** after download

That model is **Gemma 4 E4B**.


---

## Why Gemma 4 E4B is the Right Model

| Requirement | Why Gemma 4 E4B |
|---|---|
| **Offline** | 4B parameters — runs on CPU via Ollama, no GPU or internet needed |
| **Multimodal** | Native vision — understands photos directly without a separate vision model |
| **140+ languages** | Marathi and Hindi are first-class, including Devanagari script |
| **Edge-capable** | Quantized models fit on consumer hardware (~4 GB RAM) |
| **Open weights** | Apache 2.0 license — can be deployed on-device without usage fees |
| **Safe** | Instruction-tuned for safe, helpful responses — critical for healthcare |

No other open model in mid-2026 combines all five properties simultaneously.


---

## Setup

### Option A — Local Ollama (fully offline, recommended)

```bash
# 1. Install Ollama: https://ollama.com/download
# 2. Pull the model (~4 GB)
ollama pull gemma4:e4b
# 3. Start the server
ollama serve
```

### Option B — Kaggle Notebook (cloud, via HF Inference API)
Set your `HF_TOKEN` secret in Kaggle notebook settings, then run the cells below.


In [ ]:
# Install dependencies
!pip install -q gradio>=4.44.0 ollama>=0.3.0 faster-whisper>=1.0.3 \
                pillow>=10.0.0 requests>=2.31.0 python-dotenv>=1.0.0 \
                huggingface-hub>=0.24.0

In [ ]:
import os
import base64
import io
import requests
from PIL import Image, ImageDraw

# Detect environment
USE_OLLAMA = os.environ.get("USE_OLLAMA", "true").lower() == "true"
OLLAMA_BASE_URL = os.environ.get("OLLAMA_BASE_URL", "http://localhost:11434")
OLLAMA_MODEL = os.environ.get("OLLAMA_MODEL", "gemma4:e4b")
HF_TOKEN = os.environ.get("HF_TOKEN", "")
HF_MODEL_ID = "google/gemma-4-e4b-it"

print(f"Runtime: {'Ollama (offline)' if USE_OLLAMA else 'HF Inference API'}")

---

## The System Prompt

The entire behaviour of ASHA Sahayak is controlled by one carefully engineered system prompt. It instructs the model to:
- Respond **only in Marathi Devanagari** — no English, no Roman letters
- Use **three fixed section headers** for structured, scannable output
- **Never diagnose with certainty** — always hedge
- **Always refer serious cases** to the PHC (Primary Health Centre)
- **Never name specific medications**


In [ ]:
SYSTEM_PROMPT = """You are ASHA Sahayak, an AI assistant for rural Indian
community health workers (ASHA workers). You help with basic triage of
common conditions visible in photographs.

INPUT: You will receive a photograph of a patient's condition and
optionally a Marathi or Hindi description of symptoms.

OUTPUT RULES — these are non-negotiable:
1. Respond ONLY in simple Marathi using Devanagari script.
2. Never use English words or Roman letters in your response.
3. Never use complex medical terms. Write as if speaking to someone
   with a 10th-grade education.
4. Structure every response with exactly three sections, with these
   exact Marathi headers:

   काय दिसते आहे:
   [Brief description of what you observe in 1-2 sentences]

   तातडीची मदत:
   [Simple first-aid steps, numbered, max 4 items]

   डॉक्टरकडे जाणे आवश्यक आहे का?:
   [Either "होय, लवकर PHC मध्ये घेऊन जा" (yes, urgent) or
    "नाही, घरी काळजी घेता येईल" (no, can be managed at home),
    plus one sentence explaining why]

SAFETY RULES:
- Never diagnose with certainty. Use phrases like "हे ... असू शकते"
  (this could be...).
- For ANY of these signs, ALWAYS recommend urgent PHC referral:
  heavy bleeding, deep wounds, signs of infection (pus, red streaks,
  high fever), breathing difficulty, unconsciousness, snake bite,
  burns larger than a palm, anything affecting eyes.
- Never recommend specific medications by name. Only general advice
  like "स्वच्छ पाण्याने धुवा" (wash with clean water).
- If the image is unclear or doesn't show a medical condition, say so
  honestly in Marathi.

You are a triage aid, not a doctor. Your job is to help the ASHA worker
decide: handle at home or refer urgently."""

print("System prompt loaded. Length:", len(SYSTEM_PROMPT), "chars")

---

## Helper Functions


In [ ]:
def image_to_base64(image: Image.Image, max_dim: int = 1024) -> str:
    """Resize image to max_dim on longest side and return base64 JPEG string."""
    w, h = image.size
    if max(w, h) > max_dim:
        scale = max_dim / max(w, h)
        image = image.resize((int(w * scale), int(h * scale)), Image.LANCZOS)
    buf = io.BytesIO()
    image.save(buf, format="JPEG", quality=85)
    return base64.b64encode(buf.getvalue()).decode("utf-8")


def run_ollama(image: Image.Image, symptom_text: str = "") -> str:
    """Send image + symptoms to local Ollama Gemma 4 E4B instance."""
    b64 = image_to_base64(image)
    user_content = "रुग्णाची तपासणी करा."
    if symptom_text.strip():
        user_content += f"\n\nलक्षणे: {symptom_text.strip()}"

    payload = {
        "model": OLLAMA_MODEL,
        "system": SYSTEM_PROMPT,
        "prompt": user_content,
        "images": [b64],
        "stream": False,
    }
    resp = requests.post(f"{OLLAMA_BASE_URL}/api/generate", json=payload, timeout=120)
    resp.raise_for_status()
    return resp.json()["response"].strip()


def run_hf_inference(image: Image.Image, symptom_text: str = "") -> str:
    """Fallback: call HF Inference API (requires HF_TOKEN)."""
    b64 = image_to_base64(image)
    user_text = "रुग्णाची तपासणी करा."
    if symptom_text.strip():
        user_text += f"\n\nलक्षणे: {symptom_text.strip()}"

    headers = {"Authorization": f"Bearer {HF_TOKEN}", "Content-Type": "application/json"}
    payload = {
        "model": HF_MODEL_ID,
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {
                "role": "user",
                "content": [
                    {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{b64}"}},
                    {"type": "text", "text": user_text},
                ],
            },
        ],
        "max_tokens": 512,
    }
    url = f"https://api-inference.huggingface.co/models/{HF_MODEL_ID}/v1/chat/completions"
    resp = requests.post(url, headers=headers, json=payload, timeout=120)
    resp.raise_for_status()
    return resp.json()["choices"][0]["message"]["content"].strip()


def analyze(image: Image.Image, symptom_text: str = "") -> str:
    """Unified inference entry point — Ollama or HF based on USE_OLLAMA."""
    if USE_OLLAMA:
        return run_ollama(image, symptom_text)
    return run_hf_inference(image, symptom_text)

print("Helper functions defined.")

---

## Sample Inference — Synthetic Test Image

We create a simple synthetic image (red rectangle on white = simulated wound) and run it through the pipeline.


In [ ]:
# Create a synthetic test image
test_image = Image.new("RGB", (256, 256), color="white")
draw = ImageDraw.Draw(test_image)
draw.rectangle([64, 64, 192, 192], fill=(200, 50, 50))
test_image

In [ ]:
# Run inference
symptom_text = "रुग्णाच्या हाताला जखम झाली आहे, थोडे रक्त येत आहे."  # Patient has wound, some bleeding

response = analyze(test_image, symptom_text)
print(response)

---

## Prompt Engineering Insights

Getting consistent Marathi Devanagari output from a multilingual model required several iterations:

### What didn't work

1. **"Respond in Marathi"** alone — the model would mix in English medical terms like "wound" or "infection".
2. **Asking for bullet points** — formatting varied wildly across responses.
3. **No safety framing** — the model would sometimes say "apply Dettol" (a specific brand) or "take paracetamol".

### What worked

1. **Explicit script instruction**: `"Respond ONLY in simple Marathi using Devanagari script. Never use English words or Roman letters."` — this eliminated code-switching.

2. **Exact header strings in the prompt**: Including the literal Marathi headers (`काय दिसते आहे:`, etc.) in the system prompt made the model reliably reproduce them.

3. **Explicit uncertainty framing**: `"Use phrases like 'हे ... असू शकते'"` — giving the model the exact hedge phrase to use produced consistent, safe hedging.

4. **Education level anchor**: `"Write as if speaking to someone with a 10th-grade education"` — this reduced jargon significantly.

5. **Positive + negative referral examples**: Providing both `"होय, लवकर PHC मध्ये घेऊन जा"` and `"नाही, घरी काळजी घेता येईल"` as literal options made the referral decision crisp and binary.

### Key lesson
> For low-resource-language healthcare output, **show the model exactly what you want in the target language** — don't just describe it in English.


---

## Architecture Overview

```
┌─────────────────────────────────────────────────────────┐
│                  ASHA Worker's Device                   │
│                                                         │
│  ┌──────────┐    ┌──────────────┐    ┌───────────────┐  │
│  │  Camera  │    │    Voice     │    │  Typed text   │  │
│  │  Photo   │    │  Recording   │    │  (optional)   │  │
│  └────┬─────┘    └──────┬───────┘    └───────┬───────┘  │
│       │                 │                    │          │
│       │          ┌──────▼───────┐            │          │
│       │          │faster-whisper│            │          │
│       │          │  STT (local) │            │          │
│       │          └──────┬───────┘            │          │
│       │                 │ Devanagari text     │          │
│       └────────┬────────┘────────────────────┘          │
│                │                                        │
│         ┌──────▼────────────────────────────┐           │
│         │         Gradio UI (app.py)         │           │
│         │   image + symptom text → prompt   │           │
│         └──────────────┬────────────────────┘           │
│                        │                                │
│         ┌──────────────▼────────────────────┐           │
│         │    Ollama Server (gemma4:e4b)      │           │
│         │    Fully local — no internet       │           │
│         └──────────────┬────────────────────┘           │
│                        │ Marathi Devanagari response     │
│         ┌──────────────▼────────────────────┐           │
│         │  काय दिसते आहे: ...               │           │
│         │  तातडीची मदत: ...                 │           │
│         │  डॉक्टरकडे जाणे आवश्यक आहे का?: │           │
│         └───────────────────────────────────┘           │
└─────────────────────────────────────────────────────────┘
              All processing is on-device.
          No patient data leaves the device.
```


---

## Validating Marathi Output


In [ ]:
import unicodedata

def contains_devanagari(text: str) -> bool:
    return any(unicodedata.name(ch, "").startswith("DEVANAGARI") for ch in text)

def validate_response(response: str) -> dict:
    """Check that a response meets all Phase 3 acceptance criteria."""
    required_headers = [
        "काय दिसते आहे",
        "तातडीची मदत",
        "डॉक्टरकडे जाणे आवश्यक आहे का",
    ]
    results = {
        "has_devanagari": contains_devanagari(response),
        "non_empty": len(response.strip()) > 0,
    }
    for header in required_headers:
        results[f"has_header_{header[:10]}"] = header in response

    results["all_pass"] = all(results.values())
    return results

# Run validation on our test response
try:
    validation = validate_response(response)
    for check, passed in validation.items():
        status = "PASS" if passed else "FAIL"
        print(f"[{status}] {check}")
except NameError:
    print("Run the inference cell first to get a response.")

---

## Conclusion

ASHA Sahayak demonstrates that **Gemma 4 E4B is uniquely suited for offline, multilingual, multimodal healthcare AI** in low-resource settings:

- **Zero connectivity required** — the model and STT pipeline run entirely on-device.
- **Language-first design** — Marathi Devanagari output is enforced at the prompt level, not post-processed.
- **Safe by construction** — hedged language, PHC referral triggers, and medication bans are baked into the system prompt.
- **Reproducible** — the entire pipeline (Ollama + Gradio + faster-whisper) is open source and freely deployable.

**Impact potential:** With Gemma 4 E4B running offline on a ₹15,000 (~$180) Android tablet, a single ASHA worker can serve 1,000 villagers with AI-assisted triage — without a doctor, without connectivity, and without compromising patient privacy.

---
*Built for the Gemma 4 Good Hackathon (Kaggle, May 2026).*  
*GitHub: [asha-sahayak](https://github.com/YOUR_USERNAME/asha-sahayak)*  
*HF Space: [ASHA Sahayak Live Demo](https://huggingface.co/spaces/YOUR_USERNAME/asha-sahayak)*
